In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Hierarchical Triage Pipeline Benchmark: Probabilistic Joint Scaling vs Hard Case-Selector Routing (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook evaluates and compares two inference algorithms for the **5-Class Probabilistic Triage Pipeline** (Layer 2 Grouped Random Forest + Layer 3 Dual LightGBM Specialists) on the **Holdout Test Set** (ratios parsed strictly from `config/triage_conf.json`), benchmarked on **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**:

### Algorithm Definitions Evaluated
1. **MODE 1: Soft Probabilistic Joint Scaling**:
   - Computes continuous joint probabilities across all 5 classes:
     - $P(\text{ESI 1}) = P_2(\text{ESI 1})$
     - $P(\text{ESI 2}) = P_2(\text{ESI 2/3}) \times P(\text{ESI 2} \mid \text{ESI 2/3})$
     - $P(\text{ESI 3}) = P_2(\text{ESI 2/3}) \times P(\text{ESI 3} \mid \text{ESI 2/3})$
     - $P(\text{ESI 4}) = P_2(\text{ESI 4/5}) \times P(\text{ESI 4} \mid \text{ESI 4/5})$
     - $P(\text{ESI 5}) = P_2(\text{ESI 4/5}) \times P(\text{ESI 5} \mid \text{ESI 4/5})$
   - Selects predicted class with $\arg\max P(\text{ESI } c)$.

2. **MODE 2: Hard Case-Selector (If-Else Routing)**:
   - Uses Layer 2 Random Forest (`rf_esi23_esi45_extreme`) as a discrete **case selector**:
     - **IF** Random Forest predicts class `"1"` $\implies$ Final prediction is **ESI 1**.
     - **ELSE IF** Random Forest predicts class `"2_3"` $\implies$ Execute LightGBM Specialist 1 (ESI 2 vs 3) to decide between **ESI 2** or **ESI 3**.
     - **ELSE IF** Random Forest predicts class `"4_5"` $\implies$ Execute LightGBM Specialist 2 (ESI 4 vs 5) to decide between **ESI 4** or **ESI 5**.

### Target Benchmark Metrics Suite
Evaluates ONLY **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(ranger)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Master Hierarchical Triage Pipeline Initialized ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Dataset & Construct 35 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 35 Predictor Features
df_full <- data.frame(
  # 19 Raw Inputs
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 16 Continuous Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
# Stratified Test Partitioning (Parsed from config/triage_conf.json)
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
test_df      <- df_full[-in_train_val, ]
cat(sprintf("Holdout Test Set Ready (from config test_size=%.4f): %d rows x %d cols\n", test_size, nrow(test_df), ncol(test_df)))
cat("Natural 5-Class Target Distribution (ESI 1 to 5):\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load Saved Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
path_rf    <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
path_ds    <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
path_lgb23 <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
path_lgb45 <- file.path(deploy_dir, "lightgbm_esi45_model.rds")
cat("Loading model artifacts...\n")
art_rf    <- readRDS(path_rf)
art_ds    <- readRDS(path_ds)
art_lgb23 <- if (file.exists(path_lgb23)) readRDS(path_lgb23) else NULL
art_lgb45 <- if (file.exists(path_lgb45)) readRDS(path_lgb45) else NULL
cat("  - Layer 2 Model (Random Forest 3-Class Grouped Model) Loaded.\n")
cat("  - Layer 3A Model (LightGBM ESI 2 vs 3 Specialist) Loaded.\n")
cat("  - Layer 3B Model (LightGBM ESI 4 vs 5 Specialist) Loaded.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Run Inference for BOTH Algorithms
# ---------------------------------------------------------
# 1. Layer 2: Random Forest 3-Class Grouped Model ('2_3', '4_5', '1')
rf_train_feats <- art_rf$model$independent.variable.names
if (is.null(rf_train_feats)) rf_train_feats <- art_rf$model$forest$independent.variable.names
if (is.null(rf_train_feats)) rf_train_feats <- setdiff(names(test_df), "target_col")
cont_cols_rf   <- intersect(rf_train_feats, names(art_rf$preproc$mean))
binary_cols_rf <- setdiff(rf_train_feats, cont_cols_rf)
scaled_cont_rf <- predict(art_rf$preproc, test_df[, cont_cols_rf, drop = FALSE])
test_rf_scaled <- cbind(scaled_cont_rf, test_df[, binary_cols_rf, drop = FALSE])
test_rf_scaled <- test_rf_scaled[, rf_train_feats, drop = FALSE]
rf_raw_probs <- predict(art_rf$model, data = test_rf_scaled)$predictions
p_rf_23 <- rf_raw_probs[, "2_3"]
p_rf_45 <- rf_raw_probs[, "4_5"]
p_rf_1  <- if ("1" %in% colnames(rf_raw_probs)) rf_raw_probs[, "1"] else rf_raw_probs[, "other"]
# 2. Layer 3: Specialist Conditional Probabilities
ds_train_feats <- names(art_ds$preproc$mean)
ds_train_feats <- union(ds_train_feats, setdiff(names(test_df), c(ds_train_feats, "target_col")))
ds_train_feats <- intersect(names(test_df), ds_train_feats)
cont_cols_ds   <- intersect(ds_train_feats, names(art_ds$preproc$mean))
binary_cols_ds <- setdiff(ds_train_feats, cont_cols_ds)
scaled_cont_ds <- predict(art_ds$preproc, test_df[, cont_cols_ds, drop = FALSE])
test_ds_scaled <- cbind(scaled_cont_ds, test_df[, binary_cols_ds, drop = FALSE])
test_ds_x      <- as.matrix(test_ds_scaled[, ds_train_feats, drop = FALSE])
# Layer 3A (ESI 2 vs 3 LightGBM)
if (!is.null(art_lgb23) && art_lgb23$has_lgb) {
  p_esi2_given_23 <- predict(art_lgb23$model, test_ds_x)
} else if (!is.null(art_ds$model_lgb23)) {
  p_esi2_given_23 <- predict(art_ds$model_lgb23, test_ds_x)
} else if (!is.null(art_ds$model_lgb)) {
  p_esi2_given_23 <- predict(art_ds$model_lgb, test_ds_x)
} else {
  p_esi2_given_23 <- predict(art_ds$model_lgb, xgb.DMatrix(data = test_ds_x))
}
p_esi3_given_23 <- 1 - p_esi2_given_23
# Layer 3B (ESI 4 vs 5 LightGBM)
if (!is.null(art_lgb45) && art_lgb45$has_lgb) {
  p_esi4_given_45 <- predict(art_lgb45$model, test_ds_x)
} else if (!is.null(art_ds$model_lgb45)) {
  p_esi4_given_45 <- predict(art_ds$model_lgb45, test_ds_x)
} else if (!is.null(art_ds$model_xgb)) {
  p_esi4_given_45 <- predict(art_ds$model_xgb, test_ds_x)
} else {
  p_esi4_given_45 <- predict(art_ds$model_xgb, xgb.DMatrix(data = test_ds_x))
}
p_esi5_given_45 <- 1 - p_esi4_given_45
# ---------------------------------------------------------
# MODE 1: SOFT PROBABILISTIC JOINT SCALING ALGORITHM
# ---------------------------------------------------------
p_soft_1 <- p_rf_1
p_soft_2 <- p_rf_23 * p_esi2_given_23
p_soft_3 <- p_rf_23 * p_esi3_given_23
p_soft_4 <- p_rf_45 * p_esi4_given_45
p_soft_5 <- p_rf_45 * p_esi5_given_45
probs_soft <- cbind(p_soft_1, p_soft_2, p_soft_3, p_soft_4, p_soft_5)
colnames(probs_soft) <- c("1", "2", "3", "4", "5")
# ---------------------------------------------------------
# MODE 2: HARD CASE-SELECTOR (IF-ELSE ROUTING) ALGORITHM
# ---------------------------------------------------------
rf_pred_idx  <- apply(rf_raw_probs, 1, which.max)
rf_pred_cls  <- colnames(rf_raw_probs)[rf_pred_idx]
N <- nrow(test_df)
hard_preds <- character(N)
probs_hard <- matrix(0, nrow = N, ncol = 5)
colnames(probs_hard) <- c("1", "2", "3", "4", "5")
for (i in 1:N) {
  sel <- rf_pred_cls[i]
  if (sel == "1") {
    hard_preds[i] <- "1"
    probs_hard[i, "1"] <- p_rf_1[i]
    probs_hard[i, "2"] <- (1 - p_rf_1[i]) * 0.25
    probs_hard[i, "3"] <- (1 - p_rf_1[i]) * 0.25
    probs_hard[i, "4"] <- (1 - p_rf_1[i]) * 0.25
    probs_hard[i, "5"] <- (1 - p_rf_1[i]) * 0.25
  } else if (sel == "2_3") {
    if (p_esi2_given_23[i] >= 0.5) {
      hard_preds[i] <- "2"
    } else {
      hard_preds[i] <- "3"
    }
    probs_hard[i, "1"] <- p_rf_1[i]
    probs_hard[i, "2"] <- p_rf_23[i] * p_esi2_given_23[i]
    probs_hard[i, "3"] <- p_rf_23[i] * p_esi3_given_23[i]
    probs_hard[i, "4"] <- p_rf_45[i] * p_esi4_given_45[i]
    probs_hard[i, "5"] <- p_rf_45[i] * p_esi5_given_45[i]
  } else { # '4_5'
    if (p_esi4_given_45[i] >= 0.5) {
      hard_preds[i] <- "4"
    } else {
      hard_preds[i] <- "5"
    }
    probs_hard[i, "1"] <- p_rf_1[i]
    probs_hard[i, "2"] <- p_rf_23[i] * p_esi2_given_23[i]
    probs_hard[i, "3"] <- p_rf_23[i] * p_esi3_given_23[i]
    probs_hard[i, "4"] <- p_rf_45[i] * p_esi4_given_45[i]
    probs_hard[i, "5"] <- p_rf_45[i] * p_esi5_given_45[i]
  }
}
cat(sprintf("Soft Probabilistic Sum Check: Mean = %.6f\n", mean(rowSums(probs_soft))))
cat(sprintf("Hard Case-Selector Setup Complete for %d rows.\n", N))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark and Compare Both Algorithms (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
act_fac <- factor(test_df$target_col, levels = c("1", "2", "3", "4", "5"))
eval_algorithm <- function(probs_mat, pred_override_fac = NULL, name_str) {
  if (!is.null(pred_override_fac)) {
    pred_fac <- factor(pred_override_fac, levels = c("1", "2", "3", "4", "5"))
  } else {
    pred_idx <- apply(probs_mat, 1, which.max)
    pred_fac <- factor(colnames(probs_mat)[pred_idx], levels = c("1", "2", "3", "4", "5"))
  }
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  
  rec_by_cls     <- as.numeric(cm$byClass[, "Sensitivity"])
  spec_by_cls    <- as.numeric(cm$byClass[, "Specificity"])
  bal_acc_by_cls <- as.numeric(cm$byClass[, "Balanced Accuracy"])
  
  rec_by_cls[is.na(rec_by_cls)]         <- 0
  spec_by_cls[is.na(spec_by_cls)]       <- 0
  bal_acc_by_cls[is.na(bal_acc_by_cls)] <- 0
  
  roc_auc_by_cls <- sapply(1:5, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, probs_mat[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  macro_rec     <- mean(rec_by_cls)
  macro_spec    <- mean(spec_by_cls)
  macro_bal_acc <- mean(bal_acc_by_cls)
  macro_auc     <- mean(roc_auc_by_cls, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   5-CLASS BENCHMARK: %s\n", toupper(name_str)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Macro Recall (Sens)     : %.4f\n", macro_rec))
  cat(sprintf("  Macro Specificity       : %.4f\n", macro_spec))
  cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal_acc))
  cat(sprintf("  Macro ROC-AUC           : %.4f\n", macro_auc))
  cat(sprintf("============================================================\n\n"))
  print(cm$table)
  cat("\n\n")
  
  return(list(
    rec = macro_rec, spec = macro_spec, bal_acc = macro_bal_acc, auc = macro_auc
  ))
}
res_soft <- eval_algorithm(probs_soft, NULL, "Soft Probabilistic Joint Scaling")
res_hard <- eval_algorithm(probs_hard, hard_preds, "Hard Case-Selector (If-Else Routing)")
# Write CSV Summary Report
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
comp_report_df <- data.frame(
  Algorithm               = c("Probabilistic_Joint_Product", "Hard_Case_Selector_IfElse"),
  Macro_Recall            = round(c(res_soft$rec,     res_hard$rec), 4),
  Macro_Specificity       = round(c(res_soft$spec,    res_hard$spec), 4),
  Macro_Balanced_Accuracy = round(c(res_soft$bal_acc, res_hard$bal_acc), 4),
  Macro_ROC_AUC           = round(c(res_soft$auc,     res_hard$auc), 4)
)
write.csv(comp_report_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("Algorithm Comparison Test Report written to: reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart for Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_comp <- data.frame(
  Metric = rep(c("Recall", "Specificity", "Balanced_Acc", "ROC_AUC"), 2),
  Algorithm = c(rep("Probabilistic Joint Product", 4), rep("Hard Case-Selector (If-Else)", 4)),
  Score = c(
    res_soft$rec, res_soft$spec, res_soft$bal_acc, res_soft$auc,
    res_hard$rec, res_hard$spec, res_hard$bal_acc, res_hard$auc
  )
)
p_bar <- ggplot(metrics_comp, aes(x = Metric, y = Score, fill = Algorithm)) +
  geom_bar(stat = "identity", position = "dodge", width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.6), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_manual(values = c("Probabilistic Joint Product" = "#2b5c8f", "Hard Case-Selector (If-Else)" = "#e07a5f")) +
  labs(title = "Hierarchical Triage Pipeline Benchmark: Joint Product vs Hard Case-Selector",
       subtitle = "Comparing Inference Algorithms on Holdout Test Set",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 9.5, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "top")
ggsave(file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5.0, dpi = 300)
cat("Algorithm Comparison Bar Chart saved to: plots/combined_pipeline_metrics_barchart.png\n")
p_bar